[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SaronJenish/zawdwadw/blob/main/Experiment_30_Generative_Model_Output_Comparison_and_Evaluation.ipynb)


# Experiment 30: Generative Model Output Comparison and Evaluation

CSA47 – Deep Learning. Run the setup cell first, then execute the experiment cells in order.


In [ ]:
# Common setup – run this cell first
import os, time, warnings, math
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, mean_absolute_error, mean_squared_error)
np.random.seed(42)
plt.rcParams["figure.figsize"] = (7, 4)
print("Setup complete")

## Experiment 30 – Generative Model Output Comparison and Evaluation

In [ ]:
# Evaluate generated Fashion-MNIST images from Experiment 29.
# This cell expects `samples` (VAE) and `gen_samples` (GAN) to exist.
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import pairwise_distances
vae_imgs=samples.squeeze(); gan_imgs=gen_samples.squeeze()
# Use a lightweight classifier trained on real Fashion-MNIST as an Inception-Score analogue
(xreal,yreal),_=tf.keras.datasets.fashion_mnist.load_data()
xreal=xreal[:15000].astype("float32")/255.; yreal=yreal[:15000]
clf=tf.keras.Sequential([tf.keras.layers.Input((28,28)),tf.keras.layers.Flatten(),tf.keras.layers.Dense(128,activation="relu"),tf.keras.layers.Dense(10,activation="softmax")])
clf.compile("adam","sparse_categorical_crossentropy",metrics=["accuracy"])
clf.fit(xreal,yreal,epochs=3,batch_size=128,validation_split=.1,verbose=0)
def metrics(imgs):
    probs=clf.predict(imgs,verbose=0); marginal=probs.mean(0)
    score=np.exp(np.mean(np.sum(probs*(np.log(probs+1e-10)-np.log(marginal+1e-10)),axis=1)))
    labels=probs.argmax(1); diversity=len(np.unique(labels))/10
    return score,diversity,np.bincount(labels,minlength=10)
for name,imgs in [("VAE",vae_imgs),("GAN",gan_imgs)]:
    score,div,counts=metrics(imgs)
    print(name,"classifier-based IS analogue:",score,"class diversity:",div,"class counts:",counts)
# Compare sample feature distributions using PCA
real_small=xreal[:500].reshape(500,-1)
fake_small=np.vstack([vae_imgs.reshape(10,-1),gan_imgs.reshape(10,-1)])
combined=np.vstack([real_small[:100],fake_small])
projection=PCA(n_components=2).fit_transform(combined)
plt.scatter(projection[:100,0],projection[:100,1],label="Real",alpha=.5)
plt.scatter(projection[100:110,0],projection[100:110,1],label="VAE")
plt.scatter(projection[110:,0],projection[110:,1],label="GAN")
plt.legend(); plt.title("PCA: real vs generated"); plt.show()
print("FID requires a stable feature extractor and larger sample count; this compact notebook reports an IS analogue and PCA feature comparison.")